In [1]:
# notebooks/01_data_generation.ipynb — CELL 1

import pandas as pd
import numpy as np
import os

os.makedirs('../data', exist_ok=True)

np.random.seed(42)

# -----------------------------
# DATASET SIZE
# -----------------------------
N = 40000

print("Generating FD-Backed Loan dataset...")

# -----------------------------
# CUSTOMER FEATURES
# -----------------------------
age = np.random.randint(25, 70, N)

income_mo = np.random.lognormal(
    mean=10.5,
    sigma=0.8,
    size=N
).round(0)

cibil = np.clip(
    np.random.normal(715, 70, N),
    300,
    900
).astype(int)

employment = np.random.choice(
    ['Salaried', 'Self-Employed', 'Retired', 'Business'],
    N,
    p=[0.50, 0.25, 0.15, 0.10]
)

city_tier = np.random.choice(
    ['Tier1', 'Tier2', 'Tier3'],
    N,
    p=[0.35, 0.40, 0.25]
)

# -----------------------------
# FD DETAILS
# -----------------------------
fd_amount = np.random.lognormal(
    12.5,
    0.8,
    N
).round(-3).clip(50000, 5000000).astype(int)

fd_tenure_mo = np.random.choice(
    [12, 24, 36, 48, 60, 84],
    N,
    p=[0.15, 0.20, 0.25, 0.20, 0.12, 0.08]
)

fd_interest_rate = np.random.uniform(
    6.5,
    8.5,
    N
).round(2)

fd_bank = np.random.choice(
    ['SBI', 'HDFC', 'ICICI', 'Axis', 'Others'],
    N,
    p=[0.25, 0.20, 0.20, 0.15, 0.20]
)

fd_months_to_maturity = np.random.randint(
    3,
    fd_tenure_mo,
    N
).clip(1, 84)

# -----------------------------
# LOAN DETAILS
# -----------------------------
ltv = np.random.uniform(
    0.40,
    0.95,
    N
).round(3)

loan_amt = (fd_amount * ltv).round(-3).astype(int)

loan_tenure_mo = np.minimum(
    np.random.choice(
        [12, 24, 36, 48],
        N,
        p=[0.25, 0.35, 0.25, 0.15]
    ),
    fd_months_to_maturity
).clip(6, 48)

loan_interest = np.where(
    ltv > 0.80,
    fd_interest_rate + np.random.uniform(2.0, 3.5, N),
    fd_interest_rate + np.random.uniform(0.5, 2.0, N)
).round(2)

# -----------------------------
# EMI CALCULATION
# -----------------------------
emi_rate = loan_interest / (12 * 100)

emi = (
    loan_amt * emi_rate * (1 + emi_rate) ** loan_tenure_mo
    /
    ((1 + emi_rate) ** loan_tenure_mo - 1)
).round(0)

emi_income = (emi / income_mo).round(3)

# -----------------------------
# CREDIT HISTORY
# -----------------------------
num_existing_loans = np.random.poisson(1.2, N).clip(0, 6)

missed_pmts_hist = np.random.poisson(0.2, N).clip(0, 4)

pre_closure_risk = np.random.binomial(1, 0.12, N)

relationship_yrs = np.random.randint(0, 25, N)

digital_user = np.random.binomial(1, 0.68, N)

# -----------------------------
# DEFAULT PROBABILITY
# -----------------------------
default_prob = (
    0.02
    + 0.08 * (ltv > 0.85).astype(float)
    + 0.06 * (emi_income > 0.35).astype(float)
    + 0.05 * (cibil < 650).astype(float)
    - 0.03 * (cibil > 750).astype(float)
    + 0.06 * (missed_pmts_hist > 0).astype(float)
    + 0.04 * pre_closure_risk.astype(float)
    - 0.01 * (relationship_yrs > 5).astype(float)
    + 0.03 * (employment == 'Self-Employed').astype(float)
    + np.random.normal(0, 0.02, N)
).clip(0.005, 0.50)

defaulted = (
    np.random.uniform(0, 1, N) < default_prob
).astype(int)

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
df = pd.DataFrame({
    'loan_id': [f"FD{i:08d}" for i in range(N)],
    'age': age,
    'income_mo': income_mo.astype(int),
    'cibil': cibil,
    'employment': employment,
    'city_tier': city_tier,
    'fd_amount': fd_amount,
    'fd_tenure_mo': fd_tenure_mo,
    'fd_interest_rate': fd_interest_rate,
    'fd_bank': fd_bank,
    'fd_months_to_maturity': fd_months_to_maturity,
    'loan_amt': loan_amt,
    'ltv': ltv,
    'loan_tenure_mo': loan_tenure_mo,
    'loan_interest': loan_interest,
    'emi': emi.astype(int),
    'emi_income': emi_income,
    'num_existing_loans': num_existing_loans,
    'missed_pmts_hist': missed_pmts_hist,
    'pre_closure_risk': pre_closure_risk,
    'relationship_yrs': relationship_yrs,
    'digital_user': digital_user,
    'defaulted': defaulted
})

# -----------------------------
# SAVE CSV
# -----------------------------
df.to_csv('../data/fd_loans.csv', index=False)

# -----------------------------
# OUTPUT
# -----------------------------
print(f"Shape: {df.shape}")
print(f"Default rate: {defaulted.mean()*100:.2f}%")
print(f"Avg LTV: {ltv.mean():.3f}")
print(f"Avg FD amount: Rs {fd_amount.mean():,.0f}")
print(f"Avg loan amount: Rs {loan_amt.mean():,.0f}")

print("\nSaved: data/fd_loans.csv")

Generating FD-Backed Loan dataset...
Shape: (40000, 23)
Default rate: 8.33%
Avg LTV: 0.675
Avg FD amount: Rs 370,104
Avg loan amount: Rs 249,457

Saved: data/fd_loans.csv
